## Phase 01: Data Cleaning & Normalization




### Introduction
In this phase, the raw SMS Spam dataset is loaded, explored, and cleaned to ensure data quality before applying any machine learning models. Proper data preprocessing is a critical step in text classification tasks, as the performance of models highly depends on the quality and consistency of the input data.

The main objectives of this phase are:
- To inspect the structure of the raw dataset
- To handle missing values and duplicate records
- To remove irrelevant or empty columns
- To prepare a clean and well-structured dataset for further analysis and modeling

This phase lays the foundation for the subsequent stages of exploratory data analysis, feature extraction, and model development.

### 1_1. Load the CSV File and Examine Its Structure
In this step, the dataset is loaded and its basic structure is examined to understand the available features, data types, and overall size of the data.


In [33]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd

current_dir = os.getcwd()
file_path = os.path.abspath(os.path.join(current_dir, "..", "data", "raw", "spam-sms-classification-using-nlp.csv"))

print(f"📂 Looking for file at: {file_path}")

if os.path.exists(file_path):
    df = pd.read_csv(file_path, encoding='utf-8-sig') 
    print(" Data loaded successfully!")
    print(f"   Shape: {df.shape}")
    
else:
    alt_path = os.path.abspath(os.path.join(current_dir, "data", "raw", "spam-sms-classification-using-nlp.csv"))
    
    if os.path.exists(alt_path):
        df = pd.read_csv(alt_path, encoding='utf-8-sig')
        print(f" Data loaded from root path: {alt_path}")
    else:
        raise FileNotFoundError(
            f" File not found!\n"
            f"Tried path 1 (relative to notebooks): {file_path}\n"
            f"Tried path 2 (relative to root): {alt_path}\n"
            "Please check if the file exists in 'data/raw/' folder."
        )

📂 Looking for file at: c:\Users\ERAM\Desktop\Project_learning\data\raw\spam-sms-classification-using-nlp.csv
✅ Data loaded successfully!
   Shape: (5574, 2)


In [35]:
df.head()


,Class,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [36]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5574 entries, 0 to 5573
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Class    5574 non-null   object
 1   Message  5574 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [37]:
df.shape


(5574, 2)

In [38]:
df.columns


Index(['Class', 'Message'], dtype='object')

### 1_2. Checking for Missing Values, Duplicates, and Data Types
This step focuses on identifying potential data quality issues such as missing values, duplicate records, and incorrect data types.


In [39]:
# Missing values per column
print("Missing values:")
print(df.isnull().sum())
print()

# Duplicate rows count 
n_duplicates = df.duplicated().sum()
print(f"Duplicate rows: {n_duplicates}")
print()

# Data types
print("Data types:")
print(df.dtypes)


Missing values:
Class      0
Message    0
dtype: int64

Duplicate rows: 415

Data types:
Class      object
Message    object
dtype: object


**Duplicate messages:** The tables below show which messages (same text and same label) appear more than once in the dataset and how many times each is repeated.

In [40]:
# Which messages are duplicated? (group by Class and Message)
dup_counts = df.groupby(['Class', 'Message']).size().reset_index(name='count')
dup_only = dup_counts[dup_counts['count'] > 1].sort_values('count', ascending=False).reset_index(drop=True)

# Table 1: Summary
n_dup_messages = len(dup_only)
n_dup_rows = (dup_only['count'] - 1).sum() 
print("Table 1 — Summary of duplicate messages")
print("-" * 50)
print(f"  Number of unique messages that had duplicates: {n_dup_messages}")
print(f"  Total duplicate rows (extra copies): {int(n_dup_rows)}")
print()

# Table 2: List of duplicate messages (with count) — truncated text for readability
display_df = dup_only.copy()
display_df['message_short'] = display_df['Message'].apply(lambda x: (x[:60] + '…') if len(str(x)) > 60 else x)
table2 = display_df[['Class', 'count', 'message_short']]
table2

Table 1 — Summary of duplicate messages
--------------------------------------------------
  Number of unique messages that had duplicates: 290
  Total duplicate rows (extra copies): 415



,Class,count,message_short
0,ham,30,"Sorry, I'll call later"
1,ham,12,I cant pick the phone right now. Pls send a me...
2,ham,10,Ok...
3,ham,4,Your opinion about me? 1. Over 2. Jada 3. Kusr...
4,ham,4,Ok
...,...,...,...
285,ham,2,Oh k. . I will come tomorrow
286,ham,2,Ok . . now i am in bus. . If i come soon i wil...
287,ham,2,Ok i msg u b4 i leave my house.
288,ham,2,Ok lor.


In [41]:
n_before = len(df)
df = df.drop_duplicates()
n_after = len(df)
print(f"Rows before removing duplicates: {n_before}")
print(f"Rows after removing duplicates: {n_after}")
print(f"Removed: {n_before - n_after} duplicate(s)")


Rows before removing duplicates: 5574
Rows after removing duplicates: 5159
Removed: 415 duplicate(s)


### 1_3. Analyzing Label Distribution
The distribution of spam and ham messages is analyzed to understand potential class imbalance in the dataset.


In [42]:
df['Class'].value_counts()


ham     4518
spam     641
Name: Class, dtype: int64

### 1_4. Text normalization (lowercase, remove punctuation, handle special characters, HTML entities)
Text normalization techniques are applied to standardize the message content and reduce noise. **HTML entities** (e.g. `&lt;`, `&amp;`) are decoded first with `html.unescape` so that characters like `<`, `&` are then removed by regex instead of leaving artifacts such as `lt`, `amp` in the text.


In [ ]:
import re
import html

def normalize_text(text):
    # &lt; → <  & &gt; → >  & &amp; → &
    # 1. Decode HTML entities (e.g. lt&; -> <, &amp; -> &) to avoid artifacts like "lt", "gt", "amp"
    text = html.unescape(str(text))

    # 2. Convert to lowercase
    text = text.lower()

    # 3. Remove numbers
    text = re.sub(r'\d+', '', text)

    # 4. Remove punctuation and special characters (after unescape, < > & etc. are removed here)
    text = re.sub(r'[^\w\s]', '', text)

    # 5. Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text


In [44]:
df['message_normalized'] = df['Message'].apply(normalize_text)


**Tokenization (per PDF Phase 01):** Convert normalized text into a list of tokens (words) for statistical analysis in Phase 02 and compatibility with methods such as Bag of Words and TF-IDF.

### Assumptions & decisions (Phase 01)

- **Duplicates:** All duplicate rows (same `Class` and `Message`) were removed to avoid overcounting and data leakage.
- **Empty messages:** Messages that became empty or whitespace-only after normalization were removed so that no invalid samples enter training.
- **Split:** Stratified 80% train+val / 20% test, then 75% train / 25% val within train+val, so final ratios are ~60% train, ~20% val, ~20% test. Stratification keeps class balance in each split.
- **Random seed:** `random_state=42` is used for all splits to ensure reproducibility.
- **Normalization:** Lowercasing and removal of numbers/punctuation/special characters were chosen to match common text preprocessing for bag-of-words style models; HTML decoding was added to avoid artifacts (e.g. `&lt;` → "lt").

In [45]:
# Tokenization: break normalized text into list of words (tokens)
df['tokens'] = df['message_normalized'].str.split()

# Label encoding for modeling (Phase 03): ham=0, spam=1
df['label_numeric'] = df['Class'].map({'ham': 0, 'spam': 1})

print("Sample: normalized string -> tokens")
print(df[['message_normalized', 'tokens']].head(2).to_string())
print("\nLabel encoding: Class -> label_numeric")
print(df[['Class', 'label_numeric']].drop_duplicates().sort_values('label_numeric').to_string())

Sample: normalized string -> tokens
                                                                                       message_normalized                                                                                                                       tokens
0  go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat  [go, until, jurong, point, crazy, available, only, in, bugis, n, great, world, la, e, buffet, cine, there, got, amore, wat]
1                                                                                 ok lar joking wif u oni                                                                                               [ok, lar, joking, wif, u, oni]

Label encoding: Class -> label_numeric
  Class  label_numeric
0   ham              0
2  spam              1


### 1_5.Remove or handle empty messages

In [46]:
# Identify empty or whitespace-only messages (after normalization)
empty_mask = df['message_normalized'].str.strip() == ''
n_empty = empty_mask.sum()
print(f"Empty or whitespace-only messages (after normalization): {n_empty}")

# Remove such rows to avoid invalid samples in training
df = df[~empty_mask].copy()
df = df.reset_index(drop=True)
print(f"Dataset size after removing empty messages: {len(df)}")

Empty or whitespace-only messages (after normalization): 3
Dataset size after removing empty messages: 5156


### 1_6.Create train/validation/test splits


In [47]:
from sklearn.model_selection import train_test_split

# Set random seed for reproducibility
RANDOM_STATE = 42

# First split: 80% train+val, 20% test (stratified)
df_train_val, df_test = train_test_split(
    df, test_size=0.2, stratify=df['Class'], random_state=RANDOM_STATE
)

# Second split: from train+val, take 15% of original as validation (so 75% train, 25% of train_val = val)
# 0.25 * 0.8 = 0.2 of total for val -> val_size = 0.25 relative to train_val
df_train, df_val = train_test_split(
    df_train_val, test_size=0.25, stratify=df_train_val['Class'], random_state=RANDOM_STATE
)

print("Stratified split (seed=42):")
print(f"  Train:      {len(df_train)} samples ({100*len(df_train)/len(df):.1f}%)")
print(f"  Validation: {len(df_val)} samples ({100*len(df_val)/len(df):.1f}%)")
print(f"  Test:       {len(df_test)} samples ({100*len(df_test)/len(df):.1f}%)")
print("\nClass distribution in each set:")
for name, subset in [("Train", df_train), ("Validation", df_val), ("Test", df_test)]:
    print(f"  {name}: {subset['Class'].value_counts().to_dict()}")

Stratified split (seed=42):
  Train:      3093 samples (60.0%)
  Validation: 1031 samples (20.0%)
  Test:       1032 samples (20.0%)

Class distribution in each set:
  Train: {'ham': 2708, 'spam': 385}
  Validation: {'ham': 903, 'spam': 128}
  Test: {'ham': 904, 'spam': 128}


### 1_7. Data Quality Challenge: HTML Artifacts



Some messages contained **HTML entities** (e.g. `&lt;`, `&amp;`). Regex-only cleaning left fake tokens like `lt`, `amp`. 


**Solution:** decode first with `html.unescape`, then apply regex.

| Raw | Without decoding | With decoding |
| :--- | :--- | :--- |
| `"I am &lt;#&gt; inches"` | `"i am lt gt inches"` ❌ | `"i am inches"` ✅ |
| `"Me &amp; You"` | `"me amp you"` ❌ | `"me you"` ✅ |

### 1_8.Summary Report — Data Quality Issues and Solutions



This section summarizes the data quality issues and the solutions applied in Phase 01.

| Topic | Status | Action taken |
|--------|--------|------------------|
| **Missing values** | Checked | No missing values in this dataset (non-null in both columns). |
| **Duplicates** | Identified and removed | Duplicate rows removed with `drop_duplicates()`. Count removed is shown in the output above. |
| **Text normalization** | Applied | Function `normalize_text()`: first **HTML decode** (`html.unescape`) to remove artifacts like `&lt;`, `&amp;`; then lowercase, remove numbers, remove punctuation and special characters, normalize whitespace. |
| **Tokenization** | Applied | Per PDF Phase 01: normalized text converted to list of tokens (words) with `str.split()`; column `tokens`. |
| **Label encoding** | Applied | For modeling: `ham` → 0, `spam` → 1; column `label_numeric`. |
| **Empty messages** | Identified and removed | After normalization, messages that were empty or only whitespace were removed. |
| **Class imbalance** | Identified | Distribution of ham vs spam is imbalanced (ham much more frequent than spam). In later phases, appropriate metrics (e.g. F1) or sampling techniques can be used. |
| **Data split** | Done | **Stratified** split for train / validation / test with `random_state=42` for reproducibility. |

In [48]:
import os


current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
output_dir = os.path.join(project_root, "data", "processed")

os.makedirs(output_dir, exist_ok=True)

print(f"📂 Target Directory: {output_dir}")

try:
    df.to_csv(os.path.join(output_dir, 'cleaned_sms.csv'), index=False)
    if 'df_train' in globals():
        df_train.to_csv(os.path.join(output_dir, 'train.csv'), index=False)
        df_val.to_csv(os.path.join(output_dir, 'validation.csv'), index=False)
        df_test.to_csv(os.path.join(output_dir, 'test.csv'), index=False)
        
    print(" Cleaned data and splits saved successfully outside the notebook folder!")
except NameError as e:
    print(f" Error: DataFrames not found. Make sure df_train, df_val, df_test are defined. ({e})")
except Exception as e:
    print(f" Error saving files: {e}")

📂 Target Directory: c:\Users\ERAM\Desktop\Project_learning\data\processed
 Cleaned data and splits saved successfully outside the notebook folder!
